### Data Pedoman Akademik (Dense Method)

In [209]:
data_pedoman_dan_rule = {
    "data": [
        "https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/",
        "https://akademik.nurulfikri.ac.id/1-syarat-kelulusan/",
        "https://akademik.nurulfikri.ac.id/2-aturan/",
        "https://akademik.nurulfikri.ac.id/3-kode-etik-mahasiswa/",
        "https://akademik.nurulfikri.ac.id/4-suasana-akademik/",
        "https://akademik.nurulfikri.ac.id/4-profil-dosen/",
        "https://akademik.nurulfikri.ac.id/1-sejarah/",
        "https://akademik.nurulfikri.ac.id/2-administrasi/"
    ],
    "sumber_data": [
        "satuan kredit semester",
        "syarat kelulusan",
        "aturan dan kode etik",
        "kode etik mahasiswa",
        "suasana akademik",
        "profile dosen",
        "sejarah sttnf",
        "administrasi MBKM"
    ],
}

In [64]:
import os 


LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT")

In [65]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(data_pedoman_dan_rule["data"])

In [4]:
pages = []

for doc in loader.lazy_load():
    pages.append(doc)

In [ ]:
print(pages[1])

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs_splits = text_splitter.split_documents(pages)


### Model Bahasa (IndoBert)

In [12]:
# memanggil Indobert dari transformer
from transformers import BertTokenizer, AutoModel

tokenizer = BertTokenizer.from_pretrained("Indobenchmark/indobert-base-p1")
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

/Users/a/Programming/Langchain-Project/my-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
# membuat class model embdding
from typing import List 
from langchain_core.embeddings import Embeddings
import torch

class IndoBertEmbeddings(Embeddings):
    def __init__(self, model_name="indobenchmark/indobert-base-p1"):
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()


    def _generate_embedding(self, text: str) -> List[float]:
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

        with torch.no_grad():
            outputs = self.model(**inputs)

        # polling token menjadi satu vector kalimat
        token_embeddings = outputs.last_hidden_state

        # melakukan mean polling
        sentence_embeddings = token_embeddings.mean(dim=1)

        # konversi ke list python
        return sentence_embeddings.squeeze().tolist()
    

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._generate_embedding(text) for text in texts]
    

    # metode untuk pencarian query pada chroma 
    def embed_query(self, text: str) -> List[float]:
        return self._generate_embedding(text)


In [14]:
embeddings = IndoBertEmbeddings()

In [66]:
from langchain_elasticsearch import ElasticsearchStore

In [67]:
from langchain_elasticsearch import DenseVectorStrategy


vector_store = ElasticsearchStore(
    es_url="http://localhost:9200",
    index_name="langchain_index",
    embedding=embeddings,
    es_user="elastic",
    es_password="Xkhwf3uB",
    strategy=DenseVectorStrategy(hybrid=True)
)

2026-05-28 16:11:15,287 - INFO - GET http://localhost:9200/ [status:200 duration:0.015s]


In [18]:
vector_store.add_documents(docs_splits)

['19e3f3c8-d499-40f0-bdcd-c9b850875647',
 'b4a1b100-6b54-49cf-9fe3-1cda44558ee6',
 'c14b5ae9-3e2f-4496-9879-821f4091cab5',
 'a2ca53a8-a6b8-41f7-b247-5c7e3b14c2d2',
 '28083635-b9df-4d53-a9e8-fbe67b37cb59',
 '689639f4-346c-4ab5-aed9-341bccbd5fd7',
 'd2d5fe43-7189-4c31-a56b-4ab4f943764b',
 'ce81fc8a-9c5a-4e52-9ec4-8d1573d43cb2',
 'c06eae7d-933a-4770-ae2d-554ad7382bd3',
 'c2f7900f-c81e-4dc1-aa71-5e85856fed18',
 'f6d0a6ea-0ffc-4c9f-9d33-14e75a8ca250',
 'e0507855-b00c-4ef9-9fdb-6e3e5cbf4f0b',
 'eb92b77e-86b9-4819-9783-19a228c38ead',
 '851df271-4726-4815-8739-62a031265c72',
 'de98fa82-56a1-411d-ac07-9d12239714c4',
 'd615d561-bec8-4ab4-ad4f-6323639c15c7',
 'f4e4a792-0c22-4903-9843-5b84afe242c3',
 '8fd54550-a3a6-4cac-a3b8-228bffe12f2f',
 '730a51a3-8d63-4487-ba64-c11cdb97565c',
 'f423ba42-6899-4f03-9914-62b51402272f',
 'a7885895-7f16-46b2-96b6-88ef714ce0ef',
 'd2cfb45c-262d-45dd-98fb-afc8fd6c37a8',
 '0053af44-d247-4f95-ab63-6a3722b9cd48',
 '7043cff0-8c45-42b1-96d6-90451f15220b',
 'ef7bd3b0-8941-

### Testing Dense Retriever

In [19]:
retriever = vector_store.as_retriever(
    search_type="similarity", 
    search_kwargs={
        "k":5
    }
)


In [ ]:
retriever.invoke("Berapa minimal SKS untuk lulus di STT Terpadu Nurul Fikri", k=3)

In [ ]:
import json
import numpy as np


def calculate_metric(retrieved_docs, ground_truth_source, k=5):
    top_k_docs = retrieved_docs[:k]

    # ambil sumber datanya (asumsi data pedoman akademik akan punya atribut sumber data)
    retrieved_sources = [doc.metadata.get('source') for doc in top_k_docs]

    # apakah URL yang benar ada di dalam list yang ditemukan
    if ground_truth_source in retrieved_sources:
        hit_score = 1
        recall_score = 1
    else:
        hit_score = 0
        recall_score = 0

    # berapa persen dokumen di Top K yang benar
    relevant_count = retrieved_sources.count(ground_truth_source)
    precision_score = relevant_count / k
    return hit_score, precision_score, recall_score

In [ ]:
def evaluate_rag_system(dataset, retrieval_function, k_values=[1, 3, 5]):
    results = {k: {'hit_rate':[], 'precision':[], 'recall':[]} for k in k_values}

    for i, data in enumerate(dataset):
        query = data['question']
        gt_source = data['ground_truth_source']

        # query dengan vector store es
        retrieved_docs = retrieval_function.invoke(query)

        # hitung score
        for k in k_values:
            hit, prec, rec = calculate_metric(retrieved_docs, gt_source, k)

            results[k]['hit_rate'].append(hit)
            results[k]['precision'].append(prec)
            results[k]['recall'].append(rec)

    
    # rata-rata
    final_report = {}
    for k in k_values:
        final_report[f'Hit_Rate@{k}'] = np.mean(results[k]['hit_rate'])
        final_report[f'Precision{k}'] = np.mean(results[k]['precision'])
        final_report[f'Recall{k}'] = np.mean(results[k]['recall'])

    return final_report

In [ ]:
evaluasi_retrieval = [
    {
        "id": 1,
        "pertanyaan": "Apakah seorang mahasiswa S1 di STT Terpadu Nurul Fikri yang sudah menempuh perkuliahan selama 15 semester masih diperbolehkan untuk ikut wisuda dan dinyatakan lulus?",
        "jawaban": "Tidak bisa. Salah satu syarat kelulusan program Sarjana di STT Terpadu Nurul Fikri adalah masa studi maksimal tidak boleh melewati batas 7 tahun atau setara dengan 14 semester."
    },
    {
        "id": 2,
        "pertanyaan": "Berapa batas minimum indeks prestasi kumulatif serta jumlah beban studi yang wajib diselesaikan agar bisa menyelesaikan program sarjana di kampus ini?",
        "jawaban": "Mahasiswa wajib mengumpulkan beban studi sekurang-kurangnya 148 SKS (sesuai aturan prodi) dan meraih nilai indeks prestasi kumulatif (IPK) paling rendah 2.00."
    },
    {
        "id": 3,
        "pertanyaan": "Selain menyelesaikan beban Satuan Kredit Semester (SKS) dan menjaga nilai akademik tetap baik, dokumen pembuktian keahlian apa yang wajib dimiliki mahasiswa sebagai prasyarat kelulusan?",
        "jawaban": "Mahasiswa diharuskan memiliki sertifikat kompetensi untuk bisa dinyatakan lulus dari Program Sarjana."
    },
    {
        "id": 4,
        "pertanyaan": "Jika seorang mahasiswa sudah mengumpulkan 150 SKS dengan IPK 3.5, memiliki sertifikat keahlian, dan kuliah baru 4 tahun, namun ia belum mengambil skripsi, apakah dia sudah sah lulus?",
        "jawaban": "Belum sah. Mahasiswa tersebut belum memenuhi semua kriteria karena belum menyelesaikan mata kuliah Tugas Akhir (skripsi), yang merupakan salah satu syarat wajib kelulusan."
    },
    {
        "id": 5,
        "pertanyaan": "Tolong sebutkan apa saja kriteria kelayakan yang mesti dipenuhi oleh mahasiswa STT NF untuk bisa menyelesaikan studi sarjananya secara resmi!",
        "jawaban": "Kriteria kelulusannya meliputi: menyelesaikan minimal 148 SKS sesuai ketentuan prodi, lulus mata kuliah Tugas Akhir, meraih IPK minimal 2.00, mempunyai sertifikat kompetensi, dan lama waktu kuliah tidak melebihi jangka waktu 7 tahun atau 14 semester."
    },
    {
        "id": 6,
        "pertanyaan": "Kapan saja seorang mahasiswa yang terancam melampaui batas waktu kuliah akan menerima surat teguran resmi dari pihak kampus?",
        "jawaban": "Surat peringatan akan diberikan pada tiga kondisi: saat waktu studi terprogram berakhir, ketika masa kuliah sudah berjalan 6 tahun, dan saat menjelang habisnya masa perpanjangan studi."
    },
    {
        "id": 7,
        "pertanyaan": "Jika seorang mahasiswa sudah memasuki tahun keenam masa kuliahnya namun belum lulus, tindakan khusus apa yang wajib ia lakukan?",
        "jawaban": "Mahasiswa tersebut wajib menyusun agenda kegiatan untuk masa perpanjangan studi serta melaporkan progres belajarnya menggunakan formulir pemantauan dari BAAK sebagai bahan evaluasi Kaprodi."
    },
    {
        "id": 8,
        "pertanyaan": "Apakah mahasiswa yang hampir habis masa studinya otomatis mendapatkan tambahan waktu? Apa kriteria akademis yang harus dipenuhi?",
        "jawaban": "Tidak otomatis. Perpanjangan waktu hanya diberikan jika mahasiswa tinggal menyelesaikan Tugas Akhir/skripsi, bersedia menandatangani surat pernyataan bermaterai untuk mengundurkan diri jika gagal, dan wajib membuat program kerja."
    },
    {
        "id": 9,
        "pertanyaan": "Apa yang mendasari Ketua Sekolah Tinggi untuk mengambil keputusan tidak memberikan kelonggaran perpanjangan waktu bagi mahasiswa yang masa kuliahnya habis?",
        "jawaban": "Perpanjangan ditolak jika mahasiswa sulit dihubungi/tidak berada di tempat, atau secara akademis kemampuannya dinilai sudah maksimal sehingga sulit diharapkan untuk lulus."
    },
    {
        "id": 10,
        "pertanyaan": "Bagaimana alur birokrasi yang dilakukan Kaprodi jika hasil evaluasi menunjukkan seorang mahasiswa sudah tidak bisa lagi diperpanjang masa kuliahnya?",
        "jawaban": "Kaprodi akan mengajukan surat usulan pengunduran diri beserta riwayat studi kepada Ketua. Setelah SK pengunduran diri atau DO terbit, SK tersebut dikirimkan ke orang tua/wali bersama dengan transkrip nilai yang pernah ditempuh."
    },
    {
        "id": 11,
        "pertanyaan": "Setelah seorang mahasiswa resmi diputus hubungan studinya oleh kampus, apakah dia masih berhak meminta dokumen tertentu dari pihak akademik?",
        "jawaban": "Ya, mahasiswa yang bersangkutan masih diperbolehkan untuk mengajukan permohonan surat keterangan lainnya yang dianggap perlu kepada pihak kampus."
    },
    {
        "id": 12,
        "pertanyaan": "Secara regulasi kampus, apa yang dimaksud dengan jangka waktu penyelesaian beban studi dalam proses pendidikan?",
        "jawaban": "Hal tersebut merujuk pada pengertian Masa Studi, yaitu rentang waktu yang dialokasikan bagi mahasiswa untuk menuntaskan seluruh beban pembelajaran pada program studinya."
    },
    {
        "id": 13,
        "pertanyaan": "Apa status hukum akhir seorang mahasiswa S1 yang tidak mampu merampungkan seluruh kewajiban akademisnya hingga lewat dari 14 semester?",
        "jawaban": "Mahasiswa tersebut akan dinyatakan tidak mampu melanjutkan studinya dan secara resmi menyandang status Drop Out (DO)."
    },
    {
        "id": 14,
        "pertanyaan": "Rian sempat mengambil cuti kuliah selama 2 semester dan sempat tidak mengisi KRS tanpa kabar di 1 semester. Apakah total 3 semester tersebut menghentikan hitungan masa studi 7 tahunnya?",
        "jawaban": "Tidak menghentikan. Masa studi maksimal 7 tahun tetap menghitung masa cuti akademik serta periode saat mahasiswa tidak melakukan daftar ulang per semester."
    },
    {
        "id": 15,
        "pertanyaan": "Mulai kapan seorang mahasiswa akan dikenakan skema biaya kuliah berkala yang meningkat (progresif) jika ia tak kunjung lulus?",
        "jawaban": "Ketentuan SPP Progresif akan mulai diberlakukan bagi mahasiswa yang waktu kuliahnya sudah melampaui 4 tahun atau berjalan di atas 8 semester."
    },
    {
        "id": 16,
        "pertanyaan": "Bagaimana metode matematika yang diterapkan kampus untuk mengukur tingkat keberhasilan studi seorang mahasiswa, dan berapa rentang nilai maksimalnya?",
        "jawaban": "Keberhasilan studi diukur menggunakan Indeks Prestasi (IP), yang dihitung dengan membagi jumlah total hasil perkalian antara bobot SKS mata kuliah dengan nilai mata kuliah tersebut ($\sum(\text{Bobot\_SKS\_MK} \times \text{Nilai\_MK})$) dengan jumlah total SKS yang diambil. Nilai akhirnya berada pada rentang angka 0 hingga 4."
    },
    {
        "id": 17,
        "pertanyaan": "Apa dampak langsung dari hasil penilaian berkala yang dilakukan setiap akhir semester terhadap perencanaan KRS mahasiswa di periode berikutnya?",
        "jawaban": "Hasil evaluasi tersebut digunakan sebagai acuan untuk menetapkan batas maksimal beban SKS yang diperbolehkan untuk diambil oleh mahasiswa pada semester selanjutnya, dengan mempertimbangkan performa akademis dari semester yang baru lalu."
    },
    {
        "id": 18,
        "pertanyaan": "Selain dilakukan secara rutin di setiap pergantian semester, pada momen-momen krusial kapan lagikah pihak perguruan tinggi melakukan evaluasi penentu hasil studi bagi mahasiswa S1?",
        "jawaban": "Bagi program sarjana, evaluasi penentu hasil studi juga diselenggarakan pada saat akhir semester delapan (8), di bagian akhir program studi, serta pada masa menjelang habisnya batas waktu maksimal jenjang studi masing-masing."
    },
    {
        "id": 19,
        "pertanyaan": "Lembaga internal manakah di STT NF yang memegang otoritas penuh secara mandiri untuk mengusut dan menguji kasus-kasus pelanggaran aturan perilaku mahasiswa?",
        "jawaban": "Organ yang memiliki kewenangan independen untuk melakukan pemeriksaan terhadap pelanggaran Kode Etik adalah Senat Akademik Sekolah Tinggi."
    },
    {
        "id": 20,
        "pertanyaan": "Mengapa konsep etika dalam panduan ini disebut sebagai rumpun filsafat praktis? Apa esensi utamanya bagi tindakan manusia?",
        "jawaban": "Etika disebut filsafat praktis karena bertujuan untuk memberikan arahan, bimbingan, atau penyuluhan terhadap tingkah laku manusia mengenai apa saja yang semestinya dilakukan."
    },
    {
        "id": 21,
        "pertanyaan": "Apa pengertian dari sekumpulan aturan normatif yang mengikat hak serta kewajiban mahasiswa dan berfungsi sebagai tuntunan bersikap di kampus?",
        "jawaban": "Hal tersebut merupakan definisi dari Kode Etik, yaitu serangkaian norma etik berisi hak dan kewajiban sebagai kompas berfikir, bersikap, dan bertindak dalam aktivitas profesional."
    },
    {
        "id": 22,
        "pertanyaan": "Nilai dasar apa yang ditekankan dalam integritas akademik agar seluruh civitas kampus senantiasa menjunjung kejujuran dalam ranah ilmiah?",
        "jawaban": "Integritas akademik adalah nilai fundamental yang menitikberatkan pada kejujuran, tanggung jawab, serta perilaku etis dalam menjalankan setiap kegiatan ilmiah di kampus."
    },
    {
        "id": 23,
        "pertanyaan": "Siapa saja kelompok masyarakat yang dikategorikan sebagai bagian utuh dari lingkungan komunal pendidikan di STT Terpadu Nurul Fikri?",
        "jawaban": "Masyarakat tersebut dinamakan Sivitas Akademika, yang anggotanya terdiri atas dosen, tenaga kependidikan (tendik), serta mahasiswa."
    },
    {
        "id": 24,
        "pertanyaan": "Berdasarkan pedoman kampus, syarat-syarat apa saja yang harus dipenuhi dalam aktivitas pengumpulan data agar sebuah kegiatan layak disebut sebagai penelitian?",
        "jawaban": "Aktivitas tersebut harus dikerjakan secara teliti, eksplisit/jelas, sistematis, mengikuti metodologi ilmiah, serta hasilnya dapat dipertanggungjawabkan."
    },
    {
        "id": 25,
        "pertanyaan": "Bagaimana regulasi ini mendefinisikan tindakan tidak terpuji berupa klaim sepihak atas buah pikiran atau tulisan orang lain tanpa mencantumkan sumber asli?",
        "jawaban": "Tindakan tersebut dikategorikan sebagai Plagiat, yaitu mengambil atau memakai ide, opini, atau karya tulis orang lain lalu mengakuinya sebagai milik pribadi tanpa memberi kredit/atribusi kepada penciptanya."
    },
    {
        "id": 26,
        "pertanyaan": "Apakah aturan sopan santun dan tata perilaku mahasiswa ini hanya berlaku saat mereka berada di dalam area kampus Nurul Fikri saja?",
        "jawaban": "Tidak. Berdasarkan Pasal 2, Kode Etik Mahasiswa dimaksudkan sebagai pedoman beretika baik di dalam lingkungan Sekolah Tinggi maupun saat berbaur di tengah masyarakat luas."
    },
    {
        "id": 27,
        "pertanyaan": "Sebutkan profil karakter ideal mahasiswa yang ingin dibentuk oleh kampus melalui penyusunan regulasi moral ini!",
        "jawaban": "Tujuannya adalah untuk mencetak profil mahasiswa yang memiliki ketakwaan, kedalaman ilmu (berilmu), serta keluhuran budi pekerti (berakhlak mulia)."
    },
    {
        "id": 28,
        "pertanyaan": "Apa dampak suasana lingkungan belajar yang ingin diwujudkan oleh institusi dengan diterapkannya aturan kedisiplinan ini?",
        "jawaban": "Diharapkan dapat menciptakan iklim institusi pendidikan yang berjalan tertib, teratur, disertai dengan atmosfer akademik yang kondusif bagi proses belajar mengajar."
    },
    {
        "id": 29,
        "pertanyaan": "Apa saja kegunaan praktis dari adanya dokumen pedoman perilaku ini bagi pola interaksi sosial mahasiswa dan pemeliharaan fasilitas di area STT NF?",
        "jawaban": "Manfaatnya meliputi penyediaan aturan dalam pergaulan antar-mahasiswa maupun dengan sivitas akademika lainnya, serta berfungsi untuk menjaga atmosfer, lingkungan, dan fasilitas/sarana prasarana kampus."
    },
    {
        "id": 30,
        "pertanyaan": "Klaim atau privilese apa saja yang berhak didapatkan oleh seorang individu yang terdaftar kuliah di institusi ini terkait layanan mutu dan sarana fisik?",
        "jawaban": "Mahasiswa berhak memperoleh pelayanan edukasi yang selaras dengan standar pendidikan tinggi, mendapatkan informasi seputar aktivitas akademis/fasilitas, serta mempergunakan sarana kampus secara bertanggung jawab."
    },
    {
        "id": 31,
        "pertanyaan": "Selain dituntut untuk taat pada regulasi internal kampus, sistem hukum dan prinsip keilmuan apa lagi yang wajib dipatuhi mahasiswa secara makro?",
        "jawaban": "Mahasiswa wajib mematuhi norma hukum yang berlaku di NKRI, menghormati nilai-nilai kesopanan/kesusilaan, serta menjunjung tinggi kebebasan akademik beserta etos iptek yang bersifat universal, objektif, dan kritis."
    },
    {
        "id": 32,
        "pertanyaan": "Bagaimana cara konkret yang harus ditunjukkan mahasiswa dalam menghargai hak berpendapat dosen atau sesama kolega saat berada di dalam forum ilmiah seperti seminar?",
        "jawaban": "Berdasarkan Pasal 6, caranya adalah dengan menghormati kebebasan menyampaikan pikiran sesuai kaidah keilmuan, menghargai opini orang lain, tidak memaksakan kehendak, serta tidak mengutamakan ego pribadi atau kelompok."
    },
    {
        "id": 33,
        "pertanyaan": "Tanggung jawab moral apa yang diemban oleh mahasiswa STT NF dalam upaya mengembangkan dan menggaungkan produk sains serta seni di kampus?",
        "jawaban": "Mahasiswa diwajibkan ikut serta dalam memajukan dan menyebarluaskan ilmu pengetahuan, teknologi, dan seni melalui ruang kajian, penelitian, atau pembahasan ilmiah yang berlandaskan kaidah keilmuan."
    },
    {
        "id": 34,
        "pertanyaan": "Prinsip kebangsaan dan batasan moral apa saja yang wajib diperhatikan mahasiswa saat menyuarakan aspirasi atau pemikirannya ke publik?",
        "jawaban": "Setiap penyampaian opini harus menghormati hak dan hasil pemikiran orang lain, dilakukan dengan cara yang santun, selaras dengan norma hukum dan agama, serta wajib memelihara persatuan dan kesatuan bangsa."
    },
    {
        "id": 35,
        "pertanyaan": "Bagaimana indikator komunikasi yang baik ketika seorang mahasiswa berdialog atau berdiskusi, baik melalui lisan maupun tulisan?",
        "jawaban": "Komunikasi harus dibangun menggunakan tutur kata yang sopan dan santun, kepala dingin (tidak emosional), berpikir secara jernih, serta menjaga sensitivitas agar tidak menyinggung perasaan mitra bicara."
    },
    {
        "id": 36,
        "pertanyaan": "Apakah diperbolehkan secara etika jika seorang mahasiswa mengunggah data atau informasi personal temannya ke media sosial tanpa konfirmasi terlebih dahulu?",
        "jawaban": "Tidak diperbolehkan. Menurut Pasal 9, mahasiswa dilarang menyebarkan informasi pribadi milik orang lain di dunia maya tanpa mengantongi izin dari individu yang bersangkutan."
    },
    {
        "id": 37,
        "pertanyaan": "Sikap apa yang harus diambil mahasiswa saat menerima kabar yang belum jelas kebenarannya agar tidak memicu kegaduhan di masyarakat?",
        "jawaban": "Mahasiswa wajib menahan diri untuk tidak menyebarluaskan informasi yang keabsahannya belum pasti, tidak berbasis fakta, atau berpotensi melahirkan keresahan publik."
    },
    {
        "id": 38,
        "pertanyaan": "Ketika berselancar di internet dan memanfaatkan teknologi digital untuk tugas, bagaimana kewajiban mahasiswa terhadap produk digital milik orang lain?",
        "jawaban": "Mahasiswa diwajibkan untuk senantiasa menghormati serta menghargai setiap hak kekayaan intelektual (HAKI) atau karya milik orang lain."
    },
    {
        "id": 39,
        "pertanyaan": "Mengapa pihak STT Terpadu Nurul Fikri menganggap penciptaan iklim lingkungan belajar yang kondusif sebagai pilar yang sangat krusial bagi mahasiswa?",
        "jawaban": "Atmosfer akademik yang kondusif dinilai mutlak diperlukan agar proses belajar mengajar dapat berjalan secara maksimal, sehingga mampu bermuara pada capaian hasil pendidikan yang optimal."
    },
    {
        "id": 40,
        "pertanyaan": "Apa hakikat dari otonomi keilmuan yang diberikan kepada sivitas akademika dan apa hubungannya dengan masa depan sektor teknologi?",
        "jawaban": "Otonomi keilmuan adalah bentuk kemandirian dan kebebasan sivitas akademika untuk mengeksplorasi, memajukan, menyuarakan, serta mempertahankan kebenaran ilmiah demi menjamin keberlanjutan perkembangan teknologi informasi di masa mendatang."
    },
    {
        "id": 41,
        "pertanyaan": "Dalam penyusunan rencana studi atau kurikulum di STT NF, aspek non-akademis dan orientasi masa depan apa saja yang diwadahi bagi para mahasiswa?",
        "jawaban": "Kurikulum dirancang untuk memberi ruang bagi mahasiswa dalam mengasah keahlian ilmiah, membentuk keterampilan interpersonal (soft skill), serta berorientasi pada kesiapan karier dan kemudahan memperoleh pekerjaan yang relevan."
    },
    {
        "id": 42,
        "pertanyaan": "Sejauh mana kebebasan yang dimiliki oleh seorang staf pengajar (dosen) di kampus ini untuk memodifikasi rancangan pembelajaran dari mata kuliah yang diampunya?",
        "jawaban": "Dosen diberikan kelonggaran dan keleluasaan untuk memperbarui dan mengembangkan Satuan Acara Perkuliahan (SAP) atau silabus, dengan syarat harus mendapatkan persetujuan dari Ketua Program Studi terkait."
    },
    {
        "id": 43,
        "pertanyaan": "Infrastruktur digital dan fasilitas edukasi apa saja yang disediakan lembaga untuk menyokong aktivitas riset, diskusi ilmiah, serta transfer ilmu di luar kelas?",
        "jawaban": "Institusi menyediakan fasilitas fisik untuk riset, seminar, praktikum, dan workshop, serta menyediakan platform digital berupa e-learning yang dilengkapi fitur penugasan dan forum diskusi keilmuan untuk dosen dan mahasiswa."
    },
    {
        "id": 44,
        "pertanyaan": "Kelonggaran apa yang didapatkan dosen dalam mengajar kelas, dan peluang apa yang ditawarkan institusi untuk meningkatkan kapasitas keilmuan mereka?",
        "jawaban": "Dosen diberi kebebasan mengembangkan metode ajar yang inovatif dan efektif sesuai SAP/silabus. Mereka juga diberikan kesempatan mengikuti berbagai pelatihan, seminar, workshop, hingga lokakarya, baik di internal maupun eksternal kampus."
    },
    {
        "id": 45,
        "pertanyaan": "Bagaimana cara STT NF menjamin keterbukaan sistem penilaian sejak awal perkuliahan, dan apa saja aspek yang menjadi tolok ukur nilai mahasiswa?",
        "jawaban": "Transparansi dilakukan lewat penyampaian kontrak belajar di awal semester. Komponen penilaiannya meliputi tingkat kehadiran, tugas (mandiri/kelompok), kuis, serta perolehan nilai Ujian Tengah Semester (UTS) dan Ujian Akhir Semester (UAS)."
    },
    {
        "id": 46,
        "pertanyaan": "Apa yang dapat dilakukan oleh seorang mahasiswa apabila ia menemukan adanya kekeliruan atau ketidaksesuaian pada penginputan nilai ujian atau presensinya?",
        "jawaban": "Mahasiswa berhak memeriksa seluruh berkas absensi, tugas, kuis, serta hasil UTS dan UAS. Jika ada kesalahan prosedur penilaian, mahasiswa bisa berkonsultasi atau mengonfirmasi langsung kepada dosen pengampu mata kuliah tersebut."
    },
    {
        "id": 47,
        "pertanyaan": "Berapa kali interaksi tatap muka minimal yang wajib dijalani mahasiswa bersama Dosen Pembimbing Akademik (PA) dalam satu periode semester?",
        "jawaban": "Ketua program studi menyediakan jalur konsultasi di mana mahasiswa diwajibkan melakukan bimbingan dengan Dosen Pembimbing Akademik (PA) sekurang-kurangnya 3 kali dalam setiap semester."
    },
    {
        "id": 48,
        "pertanyaan": "Bagaimana alur yang harus dilewati oleh seorang dosen sebelum rencana riset atau pengabdian masyarakatnya disetujui dan didanai oleh kampus?",
        "jawaban": "Dosen harus mengikuti petunjuk pelaksanaan dari LPPM, di mana setiap usulan (proposal) penelitian dan pengabdian wajib dipresentasikan terlebih dahulu di hadapan tim LPPM sebelum anggarannya disetujui untuk dilaksanakan."
    },
    {
        "id": 49,
        "pertanyaan": "Media atau instrumen apa saja yang disediakan oleh program studi agar mahasiswa bisa memberikan rapor penilaian dan komplain terhadap tata kelola kampus?",
        "jawaban": "Mahasiswa bisa mengisi kuesioner evaluasi pembelajaran di akhir semester. Komplain juga bisa disalurkan melalui momen perwalian, survei pengajaran, email, serta akun media sosial resmi yang disediakan."
    },
    {
        "id": 50,
        "pertanyaan": "Jika mahasiswa STT NF ingin mengasah potensi non-akademik seperti jiwa kepemimpinan, olahraga, seni, atau mendalami tech group, wadah apa yang bisa mereka ikuti?",
        "jawaban": "Mahasiswa bisa bergabung dengan Unit Kegiatan Mahasiswa (UKM) yang berfokus pada IT, soft skill, olahraga, seni, dan kepemimpinan, atau terlibat aktif dalam kelompok studi/research group berbentuk IT Club."
    },
    {
        "id": 51,
        "pertanyaan": "Syarat dan peran apa yang ditawarkan kepada mahasiswa untuk membantu dosen dalam proses belajar mengajar, khususnya pada sesi praktikum?",
        "jawaban": "Mahasiswa yang memenuhi kualifikasi diberikan kesempatan untuk menjadi Asisten Dosen (Asdos) pada mata kuliah tertentu, terutama untuk mendampingi jalannya praktik di laboratorium komputer."
    },
    {
        "id": 52,
        "pertanyaan": "Bagaimana bentuk sinergi antara dosen dan mahasiswa dalam menghasilkan karya ilmiah, dan apa yang menjadi acuan utama dari kemitraan tersebut?",
        "jawaban": "Kolaborasi dapat berupa keterlibatan mahasiswa dalam proyek riset dosen, atau sebaliknya, dosen membimbing Tugas Akhir mahasiswa. Rujukan utama dari kemitraan ini adalah substansi penelitian yang mampu memberikan inspirasi bagi kedua belah pihak."
    },
    {
        "id": 53,
        "pertanyaan": "Berikan contoh nyata kegiatan rumpun teknologi yang sering dilakukan secara kolaboratif oleh dosen dan mahasiswa untuk mengabdi kepada masyarakat sekitar!",
        "jawaban": "Dosen dan mahasiswa kerap bersinergi dalam menyelenggarakan workshop dan pelatihan di bidang teknologi informasi (IT) yang ditujukan bagi para siswa serta guru-guru SMA/SMK di lingkungan terdekat kampus."
    },
    {
        "id": 54,
        "pertanyaan": "Di mana dosen pembimbing dapat mengunduh atau mengunggah dokumen pelaporan perkembangan mahasiswa MBKM, dan apa kegunaan utama repositori tersebut?",
        "jawaban": "Dosen dapat mengaksesnya melalui tautan https://bit.ly/FolderKerjaPembimbing. Folder ini berfungsi sebagai sarana penyimpanan dokumen administratif seperti jadwal pertemuan, catatan bimbingan, dan evaluasi untuk membantu memantau serta mendokumentasikan kemajuan mahasiswa."
    },
    {
        "id": 55,
        "pertanyaan": "Sebelum melakukan konversi nilai, berkas atau dokumen apa saja yang wajib diperiksa ketersediaannya oleh dosen pembimbing dari mahasiswa bimbingannya?",
        "jawaban": "Dosen pembimbing wajib memastikan kelengkapan dokumen administrasi MBKM mahasiswa yang meliputi laporan akhir, transkrip nilai, serta log harian (log kegiatan)."
    },
    {
        "id": 56,
        "pertanyaan": "Apa bentuk kontribusi administratif yang harus diberikan oleh dosen pembimbing guna memastikan legalitas dokumen kegiatan MBKM mahasiswa berjalan tanpa hambatan?",
        "jawaban": "Dosen pembimbing bertugas melayani kebutuhan administrasi mahasiswa dengan memberikan tanda tangan pengesahan pada berkas-berkas administratif, seperti laporan kegiatan dan dokumen terkait lainnya."
    },
    {
        "id": 57,
        "pertanyaan": "Siapa yang memegang tanggung jawab penuh terhadap proses input hasil penyetaraan nilai mata kuliah mahasiswa program MBKM ke dalam platform akademik kampus di setiap semester?",
        "jawaban": "Dosen pembimbing MBKM adalah pihak yang bertanggung jawab atas hasil konversi nilai mahasiswa dan bertugas menginput nilai tersebut ke dalam sistem akademik pada tiap semesternya."
    },
    {
        "id": 58,
        "pertanyaan": "Bagaimana aturan baku pembuatan dan penamaan tempat penyimpanan baru bagi mahasiswa yang mengikuti program MSIB atau kewirausahaan saat ingin mengumpulkan berkas?",
        "jawaban": "Mahasiswa harus membuat folder baru di dalam direktori program yang sesuai (MSIB, FHCI, Kewirausahaan, atau Magang Mandiri) dengan menggunakan format penamaan folder: NIM_NAMA."
    },
    {
        "id": 59,
        "pertanyaan": "Apakah mahasiswa diwajibkan secara kaku menggunakan format penyusunan laporan yang disediakan oleh STT NF jika pihak perusahaan tempat magang memiliki standar sendiri?",
        "jawaban": "Tidak bersifat kaku. Meskipun kampus menyediakan Template Laporan Kegiatan MBKM, format susunan laporan tersebut diperbolehkan untuk disesuaikan dengan standar atau format yang berlaku di mitra tempat mahasiswa magang."
    },
    {
        "id": 60,
        "pertanyaan": "Langkah apa yang harus segera diambil oleh mahasiswa setelah mereka menyelesaikan seluruh proses pengunggahan dokumen laporan dan pengisian log catatan prodi?",
        "jawaban": "Mahasiswa harus segera melakukan konfirmasi atau memberi tahu dosen pembimbing masing-masing sesaat setelah seluruh proses unggah dokumen laporan selesai dilakukan."
    },
    {
        "id": 61,
        "pertanyaan": "Bagi mahasiswa yang mengambil program magang di luar mitra Dikti, dokumen awal apa saja yang harus disiapkan dan tanda tangan seperti apa yang wajib tertera?",
        "jawaban": "Mahasiswa harus menyiapkan Surat Pernyataan Mitra dan Surat Kesanggupan. Kedua dokumen tersebut wajib dilengkapi dengan tanda tangan basah (serta cap basah khusus untuk mitra) lalu dipindai (scan)."
    },
    {
        "id": 62,
        "pertanyaan": "Jika mahasiswa memiliki dua berkas hasil scan berupa surat dari mitra dan surat pernyataan kesanggupan diri, bagaimana cara mengunggah berkas tersebut ke sistem?",
        "jawaban": "Kedua dokumen hasil pemindaian (Surat Pernyataan dan Surat Kesanggupan) tersebut harus digabungkan terlebih dahulu menjadi satu file dokumen utuh sebelum diunggah."
    },
    {
        "id": 63,
        "pertanyaan": "Ke tautan mana mahasiswa harus mengirimkan berkas pindaian persetujuan magang yang telah disatukan agar proses penyetaraan nilai bisa diproses?",
        "jawaban": "File gabungan dari Surat Pernyataan dan Surat Kesanggupan tersebut harus diunggah oleh mahasiswa ke formulir digital yang beralamat di bit.ly/form-konversi."
    },
    {
        "id": 64,
        "pertanyaan": "Bagaimana awal mula pembentukan institusi ini pada tahun 1994 sebelum bertransformasi menjadi sebuah Sekolah Tinggi Teknologi, dan kontribusi apa yang mereka berikan terkait perangkat lunak bebas pada tahun 1998?",
        "jawaban": "Sejarahnya dimulai dari berdirinya Nurul Fikri Computer & Statistics (NCS) atau NF Computer. Pada tahun 1998, lembaga ini mulai berkontribusi dengan memfasilitasi pelatihan Linux serta pemanfaatan aplikasi Open Source bagi masyarakat."
    },
    {
        "id": 65,
        "pertanyaan": "Kapan STT-NF secara legal diakui pemerintah sebagai lembaga pendidikan tinggi formal di bidang teknologi, dan apa saja jurusan yang dibuka saat awal peresmian tersebut?",
        "jawaban": "STT-NF resmi berdiri sebagai perguruan tinggi formal pada tanggal 13 Agustus 2012 melalui keputusan SK Menteri Pendidikan dan Kebudayaan No. 269/E/O/2012. Pada awal operasionalnya, kampus ini langsung membuka dua program studi, yakni Teknik Informatika dan Sistem Informasi."
    },
    {
        "id": 66,
        "pertanyaan": "Menanggapi dinamika pasar dan lonjakan sektor ekonomi digital beberapa tahun lalu, langkah ekspansif apa yang diambil oleh STT-NF terkait pilihan program pendidikan mereka?",
        "jawaban": "Untuk merespons pertumbuhan ekonomi digital yang melaju pesat, STT-NF melakukan ekspansi dengan menghadirkan program studi baru, yaitu Bisnis Digital pada tahun 2022."
    },
    {
        "id": 67,
        "pertanyaan": "Kapan batas paling lambat bagi mahasiswa untuk melunasi seluruh tanggungan uang kuliah, dan apa yang terjadi jika uang yang dikirimkan ternyata melebihi nominal yang seharusnya?",
        "jawaban": "Biaya kuliah wajib dilunasi sebelum pelaksanaan Ujian Akhir Semester (UAS). Jika terjadi kelebihan pembayaran, uang tersebut tidak dapat diambil kembali melainkan akan dialokasikan (diperhitungkan) sebagai potongan biaya kuliah pada semester berikutnya."
    },
    {
        "id": 68,
        "pertanyaan": "Mengapa kampus melarang mahasiswa menyetor uang semesteran secara tunai, dan apa keuntungan penggunaan mekanisme nomor rekening khusus dari CIMB Niaga Syariah?",
        "jawaban": "Pembayaran harus berbasis transfer menggunakan Virtual Account CIMB Niaga Syariah agar transaksi dapat langsung teridentifikasi secara otomatis oleh Bagian Keuangan dan meminimalkan risiko kekeliruan pencatatan."
    },
    {
        "id": 69,
        "pertanyaan": "Langkah awal apa saja yang harus dilewati mahasiswa jika ingin mengajukan dispensasi waktu bayar, serta persetujuan dari pihak mana saja yang wajib dilampirkan?",
        "jawaban": "Mahasiswa harus berkonsultasi terlebih dahulu dengan Dosen Pembimbing Akademik (PA), lalu mengunduh dan mengisi formulir penundaan yang di dalamnya wajib dilengkapi tanda tangan persetujuan dari orang tua/wali serta Dosen PA."
    },
    {
        "id": 70,
        "pertanyaan": "Dalam kondisi seperti apa permohonan seorang mahasiswa untuk memundurkan jadwal pembayaran uang semesternya akan ditolak atau ditangguhkan oleh bagian keuangan?",
        "jawaban": "Pengajuan penundaan pembayaran akan ditangguhkan atau tidak diproses apabila mahasiswa yang bersangkutan tercatat belum melunasi atau menyelesaikan kewajiban penundaan pembayaran dari periode sebelumnya."
    },
    {
        "id": 71,
        "pertanyaan": "Bagaimana ketentuan besaran nominal biaya bagi mahasiswa yang ingin rehat (cuti) sementara dari perkuliahan, dan kapan transaksi tersebut harus diselesaikan?",
        "jawaban": "Biaya cuti kuliah ditetapkan mengikuti tarif Biaya Semester atau Biaya Operasional Pendidikan (BOP) sesuai angkatan mahasiswa. Pembayaran wajib dirampungkan melalui transfer kode virtual account setelah mengisi formulir cuti dan sebelum periode cuti tersebut dimulai."
    },
    {
        "id": 72,
        "pertanyaan": "Mengapa lembar pembuktian bebas tunggangan finansial sangat krusial bagi mahasiswa tingkat akhir, dan di mana mereka bisa mendapatkan formulir fisiknya jika tidak mengakses tautan online?",
        "jawaban": "Surat Keterangan Lunas Biaya Pendidikan merupakan prasyarat wajib untuk mendaftar sidang skripsi/tugas akhir serta keperluan administrasi kelulusan lainnya. Jika tidak mengisi secara online, mahasiswa dapat mengambil formulir cetak langsung di ruangan BAAK Kampus B2."
    },
    {
        "id": 73,
        "pertanyaan": "Ke mana mahasiswa harus mengirimkan berkas permohonan bebas biaya kuliah yang telah diisi, dan berapa lama waktu yang dibutuhkan hingga dokumen resmi tersebut diterbitkan?",
        "jawaban": "Setelah diisi lengkap, dokumen tersebut diserahkan ke Bagian Keuangan atau dikirim melalui email ke keuangan@nurulfikri.ac.id. Pihak Keuangan akan memproses dan menerbitkan surat keterangan tersebut paling lambat dalam waktu 3 (tiga) hari kerja."
    },
    {
        "id": 74,
        "pertanyaan": "Siapa saja pihak internal kampus dan perwakilan organisasi mahasiswa yang diwajibkan hadir mengisi barisan peserta dalam prosesi pelantikan lulusan di STT NF?",
        "jawaban": "Peserta upacara wisuda terdiri atas Senat Sekolah Tinggi, para lulusan yang telah mendaftar, pejabat akademik dan struktural terkait di lingkungan STT NF, serta Ketua dan Sekretaris Badan Eksekutif Mahasiswa (BEM) STT NF."
    },
    {
        "id": 75,
        "pertanyaan": "Tahapan sidang penentu apa yang harus dilewati oleh seorang mahasiswa ganjil atau genap sebelum ia resmi menyandang status sebagai calon wisudawan?",
        "jawaban": "Mahasiswa harus terlebih dahulu dinyatakan lulus pada akhir semester ganjil/genap oleh Ketua Program Studi yang berwenang melalui tahapan Sidang Yudisium."
    },
    {
        "id": 76,
        "pertanyaan": "Di mana alamat portal digital yang harus diakses alumni untuk meregistrasikan diri ikut serta dalam seremoni kelulusan, dan kewajiban finansial apa yang mesti dituntaskan?",
        "jawaban": "Alumni harus mendaftarkan diri secara daring melalui situs https://alumni.nurulfikri.ac.id/ serta menyelesaikan pembayaran biaya wisuda melalui bank resmi yang telah ditunjuk oleh institusi."
    },
    {
        "id": 77,
        "pertanyaan": "Apakah dokumen rekam jejak non-akademik (SKPI) memiliki fungsi hukum yang sama untuk menggantikan ijazah atau transkrip nilai utama mahasiswa?",
        "jawaban": "Tidak. SKPI bukan pengganti ijazah ataupun transkrip akademik, melainkan dokumen pelengkap yang memuat capaian non-akademik mahasiswa selama kuliah untuk menunjang karier dan kelayakan kerja di dunia industri."
    },
    {
        "id": 78,
        "pertanyaan": "Kemudahan operasional dan transparansi apa saja yang ditawarkan oleh sistem administrasi SKPI digital bagi para lulusan STT-NF?",
        "jawaban": "Sistem tersebut mempermudah pengajuan administrasi secara daring, memungkinkan mahasiswa memantau status validasi dokumen mereka secara online, serta menyederhanakan proses pengumpulan data hingga penerbitan dokumen oleh pihak kampus."
    }
]

In [ ]:
report = evaluate_rag_system(dataset, retriever, [3, 5])

In [ ]:
import pprint
pprint.pprint(report)

## Pengujian Data Akademik Mahasiswa (Sparse Method)

In [68]:
data_akademik = [
    '/Users/a/Programming/Langchain-Project/external-data/sintetik-data-akademik-mahasiswa (3).xlsx'
]

### Processing document

In [70]:
from typing import Iterator
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document as LCDocument
from docling.document_converter import DocumentConverter

In [71]:
class DoclingLoader(BaseLoader):
    def __init__(self, file_path: str | list[str]) -> None:
        self._file_paths = file_path if isinstance(file_path, list) else [file_path]
        self._converter = DocumentConverter()

    def lazy_load(self) -> Iterator[LCDocument]:
        for source in self._file_paths:
            dl_doc = self._converter.convert(source).document
            text = dl_doc.export_to_markdown()
            yield LCDocument(page_content=text)

In [72]:
loader = DoclingLoader(data_akademik)

docs_akademik = loader.lazy_load()

In [73]:
data_akademik_split = text_splitter.split_documents(docs_akademik)

2026-05-28 16:16:41,136 - INFO - detected formats: [<InputFormat.XLSX: 'xlsx'>]


2026-05-28 16:16:41,201 - INFO - Going to convert document batch...
2026-05-28 16:16:41,203 - INFO - Initializing pipeline for SimplePipeline with options hash 995a146ad601044538e6a923bea22f4e
2026-05-28 16:16:41,206 - INFO - Processing document sintetik-data-akademik-mahasiswa (3).xlsx
2026-05-28 16:16:41,209 - INFO - Processing sheet 0: Sheet1
2026-05-28 16:16:41,238 - INFO - Finished converting document sintetik-data-akademik-mahasiswa (3).xlsx in 0.13 sec.


In [75]:
vector_store_sparse = ElasticsearchStore(
    es_url="http://localhost:9200",
    index_name="data-akademik-index",
    es_user="elastic",
    es_password="Xkhwf3uB",
    strategy=ElasticsearchStore.BM25RetrievalStrategy(),
)

2026-05-28 16:17:07,677 - INFO - GET http://localhost:9200/ [status:200 duration:0.005s]


In [ ]:
vector_store_sparse.add_documents(data_akademik_split)

In [84]:
vector_store_sparse.similarity_search("Siapa nama mahasiswa dengan NIM 112022001?")

2026-05-28 16:18:37,790 - INFO - POST http://localhost:9200/data-akademik-index/_search?_source_includes=metadata,text [status:200 duration:0.014s]


[Document(metadata={}, page_content='|       NIM | Nama                 | Jenis_Kelamin   | Program_Studi      |   Angkatan |   Semester |   IPK |   SKS_Lulus | Status_Akademik   | dosen_Pembimbing        |\n|-----------|----------------------|-----------------|--------------------|------------|------------|-------|-------------|-------------------|-------------------------|\n| 112022001 | Ahmad Yusuf          | L               | Teknik Informatika |       2022 |          4 |  3.75 |          80 | Aktif             | Dr. Budi Santoso M.Kom  |\n| 112022002 | Siti Aminah          | P               | Sistem Informasi   |       2022 |          4 |  3.85 |          82 | Aktif             | Rina Wati M.T.          |\n| 112022003 | Budi Setiawan        | L               | Teknik Informatika |       2022 |          4 |  3.6  |          80 | Aktif             | Dr. Budi Santoso M.Kom  |')]

In [ ]:
data_uji_akademik_mahasiswa = {
    "Siapa nama mahasiswa dengan NIM 112022001?": "Ahmad Yusuf",
    "Berapa IPK dari Siti Aminah?": "3.85",
    "Apa Program Studi Budi Setiawan?": "Teknik Informatika",
    "Berapa SKS Lulus yang dimiliki oleh Dewi Lestari?": "84",
    "Apa Status Akademik Rizky Pratama?": "Aktif",
    "Siapa dosen pembimbing Ayu Wandira?": "Rina Wati M.T.",
    "Apa jenis kelamin dari Dimas Anggara?": "L",
    "Angkatan tahun berapakah Putri Maharani?": "2022",
    "Mahasiswa dengan NIM 112022009 sedang berada di semester berapa?": "4",
    "Siapa nama mahasiswa dengan NIM 112022010?": "Nisa Rahmawati",
    "Berapa IPK dari Bayu Kurniawan?": "3.5",
    "Apa Program Studi Dina Fitriani?": "Bisnis Digital",
    "Berapa SKS Lulus yang dimiliki oleh Hendra Wijaya?": "115",
    "Apa Status Akademik Rina Sari?": "Aktif",
    "Siapa dosen pembimbing Aditya Saputra?": "Dr. Budi Santoso M.Kom",
    "Apa jenis kelamin dari Indah Permatasari?": "P",
    "Angkatan tahun berapakah Kevin Sanjaya?": "2021",
    "Mahasiswa dengan NIM 112021018 sedang berada di semester berapa?": "6",
    "Siapa nama mahasiswa dengan NIM 112021019?": "Rian Ardiansyah",
    "Berapa IPK dari Rani Mulyani?": "3.9",
    "Apa Program Studi Aris Munandar?": "Teknik Informatika",
    "Berapa SKS Lulus yang dimiliki oleh Sari Wulandari?": "140",
    "Apa Status Akademik Gilang Ramadhan?": "Aktif",
    "Siapa dosen pembimbing Wahyuni Safitri?": "Ferry Irawan M.B.A.",
    "Apa jenis kelamin dari Irvan Hakim?": "L",
    "Angkatan tahun berapakah Lia Amalia?": "2020",
    "Mahasiswa dengan NIM 112020027 sedang berada di semester berapa?": "8",
    "Siapa nama mahasiswa dengan NIM 112020028?": "Tika Yuliana",
    "Berapa IPK dari Surya Lesmana?": "3.6",
    "Apa Program Studi Vina Panduwinata?": "Sistem Informasi",
    "Berapa SKS Lulus yang dimiliki oleh Candra Wijaya?": "42",
    "Apa Status Akademik Desi Ratnasari?": "Aktif",
    "Siapa dosen pembimbing Eko Prasetyo?": "Siti Aminah M.T.",
    "Apa jenis kelamin dari Fitriani Ningsih?": "P",
    "Angkatan tahun berapakah Gatot Subroto?": "2023",
    "Mahasiswa dengan NIM 112023036 sedang berada di semester berapa?": "2",
    "Siapa nama mahasiswa dengan NIM 112023037?": "Hana Pertiwi",
    "Berapa IPK dari Iqbal Ramadhan?": "3.7",
    "Apa Program Studi Jihan Fahira?": "Bisnis Digital",
    "Berapa SKS Lulus yang dimiliki oleh Kiki Amalia?": "40",
    "Apa Status Akademik Larasati Putri?": "Aktif",
    "Siapa dosen pembimbing Mahendra Putra?": "Dr. Budi Santoso M.Kom",
    "Apa jenis kelamin dari Nanda Aulia?": "P",
    "Angkatan tahun berapakah Okan Kornelius?": "2022",
    "Mahasiswa dengan NIM 112022044 sedang berada di semester berapa?": "4",
    "Siapa nama mahasiswa dengan NIM 112022045?": "Qori Akbar",
    "Berapa IPK dari Resti Fauziah?": "3.9",
    "Apa Program Studi Satria Tama?": "Teknik Informatika",
    "Berapa SKS Lulus yang dimiliki oleh Tantri Syalindri?": "80",
    "Apa Status Akademik Umar Hapsoro?": "Aktif",
    "Siapa dosen pembimbing Vivi Haryati?": "Agus Pratama M.Kom",
    "Apa jenis kelamin dari Wira Sentosa?": "L",
    "Angkatan tahun berapakah Xena Wulandari?": "2021",
    "Mahasiswa dengan NIM 112021053 sedang berada di semester berapa?": "6",
    "Siapa nama mahasiswa dengan NIM 112021054?": "Zalfa Nabila",
    "Berapa IPK dari Andi Firmansyah?": "3.4",
    "Apa Program Studi Bella Saphira?": "Sistem Informasi",
    "Berapa SKS Lulus yang dimiliki oleh Coki Sihotang?": "122",
    "Apa Status Akademik Dinda Hauw?": "Aktif",
    "Siapa dosen pembimbing Erwin Gutawa?": "Dr. Budi Santoso M.Kom",
    "Apa jenis kelamin dari Farida Pasha?": "P",
    "Angkatan tahun berapakah Gading Marten?": "2020",
    "Mahasiswa dengan NIM 112020061 sedang berada di semester berapa?": "8",
    "Siapa nama mahasiswa dengan NIM 112020062?": "Hesti Purwadinata",
    "Berapa IPK dari Indra Bekti?": "3.45",
    "Apa Program Studi Julia Perez?": "Sistem Informasi",
    "Berapa SKS Lulus yang dimiliki oleh Kaesang Pangarep?": "144",
    "Apa Status Akademik Luna Maya?": "Aktif",
    "Siapa dosen pembimbing Mandra Naih?": "Dr. Wahyu Hidayat M.Kom",
    "Apa jenis kelamin dari Nia Ramadhani?": "P",
    "Angkatan tahun berapakah Omesh Ananda?": "2020",
    "Mahasiswa dengan NIM 112020069 sedang berada di semester berapa?": "8",
    "Siapa nama mahasiswa dengan NIM 112020070?": "Paula Verhoeven",
    "Berapa IPK dari Rafi Ahmad?": "3.55",
    "Apa Program Studi Syahnaz Sadiqah?": "Sistem Informasi",
    "Berapa SKS Lulus yang dimiliki oleh Tora Sudiro?": "36",
    "Apa Status Akademik Ussy Sulistiawaty?": "Aktif",
    "Siapa dosen pembimbing Vino Bastian?": "Dr. Budi Santoso M.Kom",
    "Apa jenis kelamin dari Wulan Guritno?": "P",
    "Angkatan tahun berapakah Yayan Ruhian?": "2023",
    "Mahasiswa dengan NIM 112023077 sedang berada di semester berapa?": "2",
    "Siapa nama mahasiswa dengan NIM 112023078?": "Zaskia Adya Mecca",
    "Berapa IPK dari Ammar Zoni?": "3.5",
    "Apa Program Studi Bunga Citra Lestari?": "Sistem Informasi",
    "Berapa SKS Lulus yang dimiliki oleh Chicco Jerikho?": "80",
    "Apa Status Akademik Dian Sastrowardoyo?": "Aktif",
    "Siapa dosen pembimbing Fedi Nuril?": "Siti Aminah M.T.",
    "Apa jenis kelamin dari Gisel Anastasia?": "P",
    "Angkatan tahun berapakah Herjunot Ali?": "2022",
    "Mahasiswa dengan NIM 112022086 sedang berada di semester berapa?": "4",
    "Siapa nama mahasiswa dengan NIM 112022087?": "Jefri Nichol",
    "Berapa IPK dari Raisa Andriana?": "3.8",
    "Apa Program Studi Nicholas Saputra?": "Teknik Informatika",
    "Berapa SKS Lulus yang dimiliki oleh Maudy Ayunda?": "88",
    "Apa Status Akademik Reza Rahadian?": "Aktif",
    "Siapa dosen pembimbing Tara Basro?": "Rina Wati M.T.",
    "Apa jenis kelamin dari Joe Taslim?": "L",
    "Angkatan tahun berapakah Chelsea Islan?": "2021",
    "Mahasiswa dengan NIM 112021094 sedang berada di semester berapa?": "6",
    "Siapa nama mahasiswa dengan NIM 112021095?": "Iko Uwais"
}

In [ ]:
from rouge_score import rouge_scorer
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_rouge_at_k(retrieved_document, ground_truth, max_k=5):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=False)
    gt_text = str(ground_truth).lower()

    
    r1_scores_at_k = {}
    r2_scores_at_k = {}
    
    max_r1 = 0.0
    max_r2 = 0.0
    
    for i in range(1, max_k + 1):
        if i <= len(retrieved_document):
            doc_content = retrieved_document[i-1].page_content.lower()
            scores = scorer.score(gt_text, doc_content)
            
            # Ambil recall secara terpisah
            r1_recall = scores['rouge1'].recall
            r2_recall = scores['rouge2'].recall
            
            if r1_recall > max_r1:
                max_r1 = r1_recall
            if r2_recall > max_r2:
                max_r2 = r2_recall
                
        r1_scores_at_k[f"@{i}"] = max_r1
        r2_scores_at_k[f"@{i}"] = max_r2
        
    return r1_scores_at_k, r2_scores_at_k

In [ ]:
# 1. Inisialisasi penampung global terpisah untuk ROUGE-1 dan ROUGE-2
all_scores_r1 = {"@1": [], "@2": [], "@3": [], "@4": [], "@5": []}
all_scores_r2 = {"@1": [], "@2": [], "@3": [], "@4": [], "@5": []}

for question, gt in data_uji_akademik_mahasiswa.items():
    # Ambil 5 dokumen dari Elasticsearch BM25
    retrieved_documents = vector_store_sparse.similarity_search(question, k=5)

    # 2. Tangkap dua output dari fungsi baru (unpacking tuple)
    query_score_r1, query_score_r2 = evaluate_rouge_at_k(retrieved_documents, gt, max_k=5)

    # 3. Masukkan skor masing-masing ke penampung yang sesuai
    for k in all_scores_r1.keys():
        all_scores_r1[k].append(query_score_r1[k])
        all_scores_r2[k].append(query_score_r2[k])


# 4. Hitung rata-rata secara terpisah
avg_score_r1 = {}
avg_score_r2 = {}

for k in all_scores_r1.keys():
    avg_score_r1[k] = round(sum(all_scores_r1[k]) / len(all_scores_r1[k]), 2)
    avg_score_r2[k] = round(sum(all_scores_r2[k]) / len(all_scores_r2[k]), 2)


# 5. Print hasil keduanya
print(f"Hasil ROUGE-1 (Recall): {avg_score_r1}")
print(f"Hasil ROUGE-2 (Recall): {avg_score_r2}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# 1. MENYIAPKAN DATA X DAN Y
# =====================================================================
# Sumbu X adalah peringkat dokumen
ks = list(avg_score_r1.keys())  # ['@1', '@2', '@3', '@4', '@5']

# Sumbu Y adalah nilai masing-masing metrik
r1_values = list(avg_score_r1.values())
r2_values = list(avg_score_r2.values())

# =====================================================================
# 2. PROSES VISUALISASI LINE CHART
# =====================================================================
sns.set_theme(style="whitegrid")  # Menggunakan background grid agar mudah dibaca
plt.figure(figsize=(9, 5.5))

# Plot Garis untuk ROUGE-1
plt.plot(
    ks, 
    r1_values, 
    marker='o',         # Titik lingkaran di setiap @K
    linewidth=2.5, 
    color='#1f77b4',    # Warna biru murni
    label='ROUGE-1'
)

# Plot Garis untuk ROUGE-2
plt.plot(
    ks, 
    r2_values, 
    marker='s',         # Titik kotak (square) di setiap @K agar beda bentuk
    linewidth=2.5, 
    color='#ff7f0e',    # Warna jingga
    label='ROUGE-2'
)

# =====================================================================
# 3. MENAMBAHKAN ANOTASI ANGKA DI SETIAP TITIK
# =====================================================================
# Anotasi untuk ROUGE-1
for i, val in enumerate(r1_values):
    plt.text(i, val + 0.02, f"{val:.2f}", ha='center', fontweight='bold', color='#1f77b4')

# Anotasi untuk ROUGE-2
for i, val in enumerate(r2_values):
    plt.text(i, val - 0.04, f"{val:.2f}", ha='center', fontweight='bold', color='#ff7f0e')

# =====================================================================
# 4. PENGATURAN LABELLING DAN ESTETIKA
# =====================================================================
# plt.title("Analisis Tren Performa Retrieval: ROUGE-1 vs ROUGE-2 dari @1 hingga @5", fontsize=13, pad=20, fontweight='bold')
plt.xlabel("Peringkat Dokumen Terambil (Top-K)", fontsize=11, labelpad=10)
plt.ylabel("Rata-rata Skor Recall", fontsize=11, labelpad=10)

plt.ylim(0.0, 1.05) # Membatasi sumbu Y dari 0 sampai maksimal skor 1 (dilebihkan dikit untuk teks)
plt.legend(loc="lower right", frameon=True, shadow=True) # Menampilkan legenda di kanan bawah

plt.tight_layout()
plt.show()

## Permodelan (LLM)

In [78]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
import operator
from langchain_core.messages import AIMessage

### Tools 

In [79]:

@tool
def query_from_academic_rule(query: str):
    """
    Gunakan tool ini untuk query dari user yang HANYA bermaksud untuk pertanyaan seputar ATURAN, KEBIJAKAN, SYARAT, atau PROSEDUR KAMPUS.
    
    args: 
        query: Search terms to look for
    """
    try:
        docs = vector_store.similarity_search(query, k=3)

        if not docs:
            return "Maaf, tidak ditemukan informasi relevan di data pedoman akademik."
        
        formatted_results = "\n\n---\n\n".join([d.page_content for d in docs])
        return f"Ditemukan informasi berikut dari Pedoman Akademik:\n{formatted_results}"
    
    except Exception as e:
        return "Terjadi Kesalahan saat mengakses vector database"


@tool
def get_student_academic_record(query: str):
    """
    Gunakan tool ini untuk mencari DATA PRIBADI MAHASISWA tertentu.
    Gunakan Atribut mahasiswa seperti (nama mahasiswa, NIM, status, Dosen Pembimbing) sebagai query pencarian.
    
    Args:
        query: Search terms to look for
    """
    try:
        docs = vector_store_sparse.similarity_search(query)

        if not docs:
            return "Maaf, tidak ditemukan informasi relevan di data akademik mahasiswa"
        
        formatted_results = "\n\n---\n\n".join([d.page_content for d in docs])
        return f"Ditemukan informasi berikut dari data akademik mahasiswa:\n{formatted_results}"
    
    except Exception as e:
        return "Terjadi kesalahn saat mengakses data akademik"

In [80]:
# Entry point (pintu masuk system)
system_prompt_cot = """
You are an intelligent Intent Router for the STT-NF Academic System. 
Your goal is to decide which tool(s) are required to answer the user's query accurately.

AVAILABLE TOOLS:
1. `query_from_academic_rule`: For General Regulations, Procedures, Requirements (Cuti, Sidang, KRS, etc).
2. `get_student_academic_record`: For Specific Student Identity (Name, NIM), Grades, IPK, Status.

OUTPUT FORMAT RULES:
1. IF you need a tool: Output a valid JSON object in the 'Action' step.
   Format: {"name": "tool_name", "args": {"argument_name": "value"}}
2. IF NO tool is needed (Greeting, Chit-chat, or OOT): Output the direct response text in **INDONESIAN**. Do not output JSON.
3. NEVER put a student's name (e.g., "Agus", "Budi") inside the `query_from_academic_rule` argument. The rulebook does NOT contain student names!

FEW-SHOT EXAMPLES:

EXAMPLE 1 (General Rule):
User: "Bagaimana prosedur mengajukan banding nilai ujian?"
Thought: The user is asking for the "prosedur" of "banding nilai". This is a general regulation.
Action: {"name": "query_from_academic_rule", "args": {"query": "prosedur banding nilai ujian"}}

EXAMPLE 2 (Specific Data):
User: "Cek status mahasiswa bernama Siti Aminah?"
Thought: The query explicitly mentions "Siti Aminah". I need to check individual status.
Action: {"name": "get_student_academic_record", "args": {"query": "Siti Aminah"}}

EXAMPLE 3 (Explicit Hybrid):
User: "Berapa jumlah SKS mahasiswa Agus Nasution, Dan dari pedoman akademik, apakah dia bisa lulus?"
Thought: User explicitly asks for specific data "SKS Agus Nasution" AND rule validation "syarat kelulusan". Need BOTH tools.
Action: [
    {"name": "get_student_academic_record", "args": {"query": "Agus Nasution"}},
    {"name": "query_from_academic_rule", "args": {"query": "syarat kelulusan SKS"}}
]

EXAMPLE 4 (Implicit Hybrid - LOGICAL DEDUCTION):
User: "Apakah Agus Nasution akan dikenakan SPP Progresif berdasarkan pedoman akademik?"
Thought: 
1. The query asks about applying a specific rule ("SPP Progresif") to a specific person ("Agus Nasution").
2. To evaluate this, I MUST know Agus's current status/semester (Requires `get_student_academic_record`).
3. I also MUST know the general rule for "SPP Progresif" (Requires `query_from_academic_rule`).
4. I must split this into two tool calls. I will NOT put Agus's name in the rule query.
Action: [
    {"name": "get_student_academic_record", "args": {"query": "Agus Nasution"}},
    {"name": "query_from_academic_rule", "args": {"query": "aturan syarat SPP Progresif"}}
]

EXAMPLE 5 (Greeting - Direct Response):
User: "Halo selamat malam"
Thought: Just a greeting. No tool needed. I must answer politely in Indonesian.
Action: Halo, selamat malam! Saya asisten akademik STT-NF. Ada yang bisa saya bantu terkait informasi akademik atau data mahasiswa?

INSTRUCTION:
Based on the logic above, determine the Thought and Action for the following user query.
"""

GENERATOR_PROMPT = """
PERAN:
Anda adalah Asisten Akademik Cerdas di kampus STT-NF (Sekolah Tinggi Terpadu Nurul Fikri).
Tugas Anda adalah menyusun jawaban akhir kepada pengguna dalam Bahasa Indonesia yang formal namun ramah.

INPUT ANDA:
1. Pertanyaan Asli User.
2. Data Mentah (JSON/Teks) dari hasil eksekusi alat (Context).

ATURAN MENJAWAB:
1. **GROUNDING:** Jawab HANYA berdasarkan data di `CONTEXT`. Jangan berhalusinasi.
2. **JIKA DATA DITEMUKAN:** Rangkum data tersebut menjadi kalimat yang enak dibaca.
   - Contoh: "Berdasarkan data, mahasiswa Tono (NIM 123) berstatus Aktif dengan IPK 3.8."
3. **JIKA DATA KOSONG/ERROR:** Katakan jujur: "Maaf, data tidak ditemukan. Mohon periksa nama atau NIM kembali."
4. **JANGAN** menampilkan struktur JSON mentah ke user.
5. **JANGAN** menyebutkan teknis internal (seperti "saya menggunakan tool get_student").

Silakan jawab pertanyaan user berdasarkan Context berikut.
"""

In [81]:
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[List[AnyMessage], add_messages]

In [82]:
import json
import re
import ast
from typing import Annotated, List, TypedDict, Optional

from langchain_core.messages import (
    AIMessage, 
    ToolMessage, 
    SystemMessage, 
    HumanMessage, 
    AnyMessage
)
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# --- 1. DEFINISI STATE ---
class AgentState(TypedDict):
    messages: Annotated[List[AnyMessage], add_messages]

# PROMPT KHUSUS GENERATOR (NODE 3) 
GENERATOR_PROMPT = """
PERAN:
Anda adalah Asisten Akademik Cerdas di kampus STT-NF (Sekolah Tinggi Terpadu Nurul Fikri).
Tugas Anda adalah menyusun jawaban akhir kepada pengguna dalam Bahasa Indonesia yang formal namun ramah.

ATURAN MENJAWAB:
1. **GROUNDING:** Jawab HANYA berdasarkan data di `CONTEXT`. Jangan berhalusinasi.
2. **KONDISI YA/TIDAK:** Pastikan jawaban Mengandung kata "YA" jika pernyataan/pertanyaan USER BENAR/MEMENUHI SYARAT, dan "TIDAK" jika pernyataan USER SALAH/TIDAK MEMENUHI SYARAT.
3. **JIKA DATA KOSONG/ERROR:** Katakan jujur: "Maaf, data tidak ditemukan. Mohon periksa nama atau NIM kembali."
4. **JANGAN** menampilkan struktur JSON mentah ke user.

PROSES BERPIKIR (WAJIB DIIKUTI SEBELUM MEMBERIKAN JAWABAN AKHIR):
Untuk menghindari kesalahan logika saat membaca data, jabarkan analisis Anda menggunakan format berikut:

[EKSTRAKSI DATA MAHASISWA]
- Tuliskan variabel yang relevan saja (contoh: IPK = ..., IPS = ..., SKS = ...). Jika tidak ada, tulis "Tidak ada".

[EKSTRAKSI ATURAN AKADEMIK]
- Tuliskan syarat aturannya (contoh: Syarat lulus IPK >= 2.00). Jika tidak ada, tulis "Tidak ada".

[ANALISIS LOGIKA]
- Bandingkan data mahasiswa dengan aturan akademik secara teliti. Pastikan Anda tidak tertukar antara IPS (Indeks Prestasi Semester) dan IPK (Indeks Prestasi Kumulatif).

[JAWABAN AKHIR]
- Tuliskan jawaban natural Anda di sini (Pastikan mengandung kata YA atau TIDAK sesuai hasil analisis).

INFORMASI DARI SISTEM (CONTEXT):
{context_data}

PERTANYAAN USER:
{user_query}
"""

# --- 3. CLASS AGENT UTAMA ---
class Agent:
    def __init__(self, model, tools, router_prompt=""):
        self.router_system = router_prompt
        self.tools = {t.name: t for t in tools}
        
        # Model 1: Router (Punya kemampuan bind tools)
        self.router_model = model.bind_tools(tools)
        
        # Model 2: Generator (Model polos untuk merangkai kata)
        self.generator_model = model 
        
        # --- DEFINISI GRAPH ---
        graph = StateGraph(AgentState)
        
        # Tambahkan Node
        graph.add_node("router", self.call_router)
        graph.add_node("tool_executor", self.take_action)
        graph.add_node("generator", self.run_generator)
        
        # Tentukan Entry Point
        graph.set_entry_point("router")
        
        # Edge Kondisional: Router -> (Tool Executor ATAU End)
        graph.add_conditional_edges( 
            "router", 
            self.exists_action, 
            {True: "tool_executor", False: END} 
        )
        
        # Edge Normal: Tool Executor -> Generator
        graph.add_edge("tool_executor", "generator")
        
        # Edge Normal: Generator -> End
        graph.add_edge("generator", END)
        
        self.graph = graph.compile()

    def _try_extract_tool_calls(self, content: str) -> List[dict]:
        if not content: return []
        
        # Bersihkan markdown
        content = re.sub(r'```json\s*', '', content)
        content = re.sub(r'```', '', content)
        
        extracted_tools = []
        
        # Coba cari format Array [...] dulu, kalau tidak ada baru cari Object {...}
        list_match = re.search(r'\[.*\]', content, re.DOTALL)
        dict_match = re.search(r'\{.*\}', content, re.DOTALL)
        
        candidates = []
        if list_match:
            candidates.append(list_match.group()) # Memasukkan [...]
        elif dict_match:
            candidates.append(dict_match.group()) # Memasukkan {...}
            
        for candidate in candidates:
            try:
                # loads akan berhasil karena formatnya sudah pasti [...] atau {...}
                data = json.loads(candidate)
                
                # Normalisasi: Jika yang didapat hanya 1 dict {...}, jadikan list [{...}]
                if isinstance(data, dict): 
                    data = [data]
                
                # Looping isi list untuk memvalidasi tools
                if isinstance(data, list):
                    for item in data:
                        if not isinstance(item, dict): continue
                        
                        name = item.get('name') or item.get('tool')
                        args = item.get('args') or item.get('arguments') or {}
                        
                        # Pastikan tool valid dan terdaftar
                        if name and name in self.tools:
                            extracted_tools.append({
                                'name': name,
                                'args': args,
                                'id': f"manual_{len(extracted_tools)}"
                            })
            except Exception as e:
                # Lanjut jika gagal parse (untuk menghindari sistem crash)
                continue 
                
        return extracted_tools

    # --- LOGIC: EXISTS ACTION ---
    def exists_action(self, state: AgentState) -> bool:
        result = state['messages'][-1]
        
        # 1. Cek Native Tool Calls
        if hasattr(result, "tool_calls") and len(result.tool_calls) > 0:
            return True
            
        # 2. Cek Manual Parsing
        if self._try_extract_tool_calls(result.content):
            print("🕵️ Valid JSON Action detected via regex.")
            return True
        
        return False

    # --- NODE 1: ROUTER ---
    def call_router(self, state: AgentState):
        messages = state["messages"]
        if self.router_system:
            if not isinstance(messages[0], SystemMessage):
                messages = [SystemMessage(content=self.router_system)] + messages
            else:
                messages[0] = SystemMessage(content=self.router_system)
        
        # 1. Panggil Model
        response = self.router_model.invoke(messages)
        content = response.content
        
        # 2. Cek apakah ini Tool Call (JSON)?
        # Kita gunakan helper yang sama untuk mendeteksi
        is_tool_call = False
        if hasattr(response, "tool_calls") and len(response.tool_calls) > 0:
            is_tool_call = True
        elif self._try_extract_tool_calls(content):
            is_tool_call = True
            
        # 3. LOGIKA CLEANING:
        # Jika BUKAN tool call (berarti Greeting/Chat biasa),
        # tapi ada format "Action:", kita potong text sebelumnya.
        if not is_tool_call and "Action:" in content:
            # Ambil teks setelah kata "Action:"
            clean_response = content.split("Action:")[-1].strip()
            
            # Update isi pesan agar user terima bersih
            response.content = clean_response
            
        return {'messages': [response]}

    # --- NODE 2: TOOL EXECUTOR ---
    def take_action(self, state: AgentState):
        llm_message = state['messages'][-1]
        tools_to_run = []
        
        if hasattr(llm_message, "tool_calls") and len(llm_message.tool_calls) > 0:
            tools_to_run = llm_message.tool_calls
        else:
            tools_to_run = self._try_extract_tool_calls(llm_message.content)

        results = []
        for tool_call in tools_to_run:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            tool_id = tool_call.get('id')

            print(f"🛠️ Executing: {tool_name} with {tool_args}") # Debug
            
            try:
                # Execute tool
                tool_output = self.tools[tool_name].invoke(tool_args)
            except Exception as e:
                tool_output = f"Error executing tool: {str(e)}"

            results.append(ToolMessage(
                tool_call_id=tool_id,
                name=tool_name,
                content=str(tool_output)
            ))

        return {"messages": results}

    # --- NODE 3: FINAL GENERATOR ---
    def run_generator(self, state: AgentState):
        messages = state['messages']
        
        # 1. Cari pertanyaan User yang asli
        user_query = "Unknown query"
        for msg in reversed(messages):
            if isinstance(msg, HumanMessage):
                user_query = msg.content
                break
        
        # 2. Siapkan Context
        context_data = ""

        for msg in reversed(messages):
            if isinstance(msg, ToolMessage):
                context_data += f"[{msg.name}]:\n{msg.content}\n\n"
            elif isinstance(msg, AIMessage):
                break
        
        if not context_data.strip():
            context_data = "Data Kosong."
            
        # 3. Buat Prompt untuk Generator
        final_prompt_content = f"""
        INFORMASI DARI SISTEM (CONTEXT):
        {context_data}
        
        PERTANYAAN USER:
        {user_query}
        """
        
        messages_for_generator = [
            SystemMessage(content=GENERATOR_PROMPT),
            HumanMessage(content=final_prompt_content)
        ]
        
        # 4. Invoke Model (Tanpa tools, pure generation)
        response = self.generator_model.invoke(messages_for_generator)
        raw_text = response.content
        
        # Bersihkan agar coret-coretan pikiran LLM tidak terlihat oleh user/sistem evaluasi
        if "[JAWABAN AKHIR]" in raw_text:
            clean_answer = raw_text.split("[JAWABAN AKHIR]")[-1].strip()
            response.content = clean_answer
            
        return {"messages": [response]}

### LLM model

In [33]:
from langchain_ollama import ChatOllama

In [34]:
model = ChatOllama(
    model="mistral:7b-instruct-v0.3-q8_0", 
    temperature=0, 
    streaming=True)

In [35]:
tools = [query_from_academic_rule, get_student_academic_record]

In [36]:
tools[1]

StructuredTool(name='get_student_academic_record', description='Gunakan tool ini untuk mencari DATA PRIBADI MAHASISWA tertentu.\nGunakan Atribut mahasiswa seperti (nama mahasiswa, NIM, status, Dosen Pembimbing) sebagai query pencarian.\n\nArgs:\n    query: Search terms to look for', args_schema=<class 'langchain_core.utils.pydantic.get_student_academic_record'>, func=<function get_student_academic_record at 0x139ef3880>)

In [37]:
bot = Agent(model, tools, router_prompt=system_prompt_cot)

In [42]:
result = bot.graph.invoke({"messages": [HumanMessage(content="siapa dosen teori bahasa otomata di sttnf")]})

2026-05-27 16:33:25,455 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 16:33:35,290 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.085s]


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: get_student_academic_record with {'query': 'Dosen Teori Bahasa Otomata'}


2026-05-27 16:33:46,749 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [43]:
print(result['messages'][-1].content)

Tidak dapat menentukan dosen teori Bahasa Otomata di STT-NF, karena data yang disediakan tidak mencakup informasi tersebut.


In [266]:
Binary_Test_Data_Akademik_Mahasiswa = [
    {
        "question": "Apakah Mega Anggraini adalah mahasiswa semester 5",
        "ground_truth_answer": "Tidak"
    },

    {
        "question": "Apakah benar mahasiswa atas nama Agus Nasution jurusan Bisnis Digital telah berstatus lulus",
        "ground_truth_answer": "Ya"
    },

    {
        "question": "Apakah mahasiswa atas nama Eka Fitriani, mahasiswa semester 10, punya IPK 1.0",
        "ground_truth_answer": "Tidak"
    },
    {
        "question": "Apakah Ir. Sigit Santoso, M.Kom adalah dosen pembimbing mahasiswa atas namaRahmat Hasanah",
        "ground_truth_answer": "Ya"
    },
    {
        "question": "Apakah mahasiswa dengan NIM 2021030039 jurusannya adalah Bisnis Digital",
        "ground_truth_answer": "Ya"
    },]

In [267]:
Binary_Test_Data_Akademik_Mahasiswa

[{'question': 'Apakah Mega Anggraini adalah mahasiswa semester 5',
  'ground_truth_answer': 'Tidak'},
 {'question': 'Apakah benar mahasiswa atas nama Agus Nasution jurusan Bisnis Digital telah berstatus lulus',
  'ground_truth_answer': 'Ya'},
 {'question': 'Apakah mahasiswa atas nama Eka Fitriani, mahasiswa semester 10, punya IPK 1.0',
  'ground_truth_answer': 'Tidak'},
 {'question': 'Apakah Ir. Sigit Santoso, M.Kom adalah dosen pembimbing mahasiswa atas namaRahmat Hasanah',
  'ground_truth_answer': 'Ya'},
 {'question': 'Apakah mahasiswa dengan NIM 2021030039 jurusannya adalah Bisnis Digital',
  'ground_truth_answer': 'Ya'}]

In [268]:
def extract_binary_label(llm_response):
    text = llm_response.lower()

    is_ya = re.search(r'\b(ya|benar|sesuai|sudah)\b', text)
    is_tidak = re.search(r'\b(tidak|bukan|belum|salah)\b', text)
    
    if is_tidak:
        return "Tidak"
    elif is_ya:
        return "Ya"
    else:
        return "Halusinasi"

def calculate_binary_metrics(y_true, y_pred):
    tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt=="Ya" and yp=="Ya") # True positive
    tn = sum(1 for yt, yp in zip(y_true, y_pred) if yt=="Tidak" and yp=="Tidak") # True Negatif
    fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt=="Tidak" and yp=="Ya") # False Postive
    fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt=="Ya" and yp=="Tidak") # False Negative

    accuracy = (tp + tn) / len(y_true) if len(y_true) > 0 else 0
    precision = tp / (tp + fp) if (tp+fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    return {
        "Accuracy": round(accuracy, 2),
        "Precision": round(precision, 2),
        "Recall": round(recall, 2)
    }


In [269]:
import pandas as pd
from IPython.display import display


def run_evaluation_binary(agent, test_data):
    results = []
    y_true = []
    y_pred = []

    print("Mulai Evaluasi")

    for i, data in enumerate(test_data):
        query = data['question']
        gt_label = data['ground_truth_answer']

        # call agent
        try:
            response = agent.graph.invoke({"messages": HumanMessage(content=query)})
            ai_message = response['messages'][-1].content

            # mengambil hasil retrieval
            retrieved_context = "tidak ada konteks"
            for msg in response['messages']:
                if hasattr(msg, 'name') and msg.type =='tool':
                    retrieved_context = msg.content
                    break
    
        except Exception as e:
            ai_message = f"Error: {str(e)}"
            retrieved_context = "error"

        pred_label = extract_binary_label(ai_message)
        status = "✅ Sesuai" if pred_label == gt_label else "❌ Meleset"

        results.append({
            "No": i + 1,
            "Pertanyaan": query,
            "Konteks Retrieval (Raw Data)": retrieved_context,
            "Jawaban Generator (AI)": ai_message,
            "Ekstrak": pred_label,
            "Target (GT)": gt_label,
            "Status": status
        })

        y_true.append(gt_label)
        y_pred.append(pred_label)

    df_results = pd.DataFrame(results)
    
    metrics = calculate_binary_metrics(y_true, y_pred)
    
    return df_results, metrics



In [270]:
df_laporan, score_metric = run_evaluation_binary(bot, Binary_Test_Data_Akademik_Mahasiswa)

Mulai Evaluasi


2026-02-19 15:51:45,904 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-19 15:51:57,891 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.040s]


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: get_student_academic_record with {'query': 'Mega Anggraini'}


2026-02-19 15:52:14,739 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-19 15:52:31,613 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-19 15:52:40,936 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.058s]


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: get_student_academic_record with {'query': 'Agus Nasution'}


...

2026-02-19 15:53:46,677 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-19 15:54:02,327 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-19 15:54:15,311 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.016s]


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: get_student_academic_record with {'query': 'Rahmat Hasanah'}


2026-02-19 15:54:32,386 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-19 15:54:48,814 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-19 15:54:58,870 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.010s]


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: get_student_academic_record with {'query': '2021030039'}


2026-02-19 15:55:03,121 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [271]:
display(df_laporan)

,No,Pertanyaan,Konteks Retrieval (Raw Data),Jawaban Generator (AI),Ekstrak,Target (GT),Status
0,1,Apakah Mega Anggraini adalah mahasiswa semester 5,Ditemukan informasi berikut dari data akademik...,"Maaf, data yang kami dapatkan tidak menunjukk...",Tidak,Tidak,✅ Sesuai
1,2,Apakah benar mahasiswa atas nama Agus Nasution...,Ditemukan informasi berikut dari data akademik...,"Ya, benar. Mahasiswa Agus Nasution dengan jur...",Ya,Ya,✅ Sesuai
2,3,"Apakah mahasiswa atas nama Eka Fitriani, mahas...",Ditemukan informasi berikut dari data akademik...,"Maaf, data yang Anda tanyakan tidak ditemukan...",Tidak,Tidak,✅ Sesuai
3,4,"Apakah Ir. Sigit Santoso, M.Kom adalah dosen p...",Ditemukan informasi berikut dari data akademik...,"Maaf, data tidak ditemukan yang menunjukkan b...",Tidak,Ya,❌ Meleset
4,5,Apakah mahasiswa dengan NIM 2021030039 jurusan...,"Maaf, tidak ditemukan informasi relevan di dat...","Maaf, data tidak ditemukan. Mohon periksa nam...",Tidak,Ya,❌ Meleset


In [272]:
print(score_metric)

{'Accuracy': 0.6, 'Precision': 1.0, 'Recall': 0.33}


In [274]:
df_laporan.to_excel("testing.xlsx")

### Pengujian Data Pedoman Akademik

In [358]:
dataset_rule = [
    {
        "question":"Apakah benar untuk lulus di STTNF harus memiliki IPK minimal 2.00",
        "ground_truth_answer":"Ya"
    },
]
[
     {
        "question":"Apakah benar Pasal 21 pada kode etik mahasiswa STTNF Etika mahasiswa dalam bidang penelitian",
        "ground_truth_answer":"Tidak"
    },
    {
        "question":"Apakah benar surat peringatan diberikan kepada mahasiswa yang mempunyai masa studi dibawah 6 tahun",
        "ground_truth_answer":"Tidak"
    },
    {
        "question":"Apakah benar di STTNF bahwa mahasiswa yang masa studinya melapui 8 semester akan diberlakukan ketentuan SPP Progresif",
        "ground_truth_answer":"Ya"
    },
    {
        "question":"Apakah benar minimal sks untuk lulus di sttnf adalah 148 SKS ",
        "ground_truth_answer":"Ya"
    }
]

[{'question': 'Apakah benar Pasal 21 pada kode etik mahasiswa STTNF Etika mahasiswa dalam bidang penelitian',
  'ground_truth_answer': 'Tidak'},
 {'question': 'Apakah benar surat peringatan diberikan kepada mahasiswa yang mempunyai masa studi dibawah 6 tahun',
  'ground_truth_answer': 'Tidak'},
 {'question': 'Apakah benar di STTNF bahwa mahasiswa yang masa studinya melapui 8 semester akan diberlakukan ketentuan SPP Progresif',
  'ground_truth_answer': 'Ya'},
 {'question': 'Apakah benar minimal sks untuk lulus di sttnf adalah 148 SKS ',
  'ground_truth_answer': 'Ya'}]

In [304]:
dataset_rule

[{'question': 'Apakah benar untuk lulus di STTNF harus memiliki IPK minimal 2.00',
  'ground_truth_answer': 'Ya'},
 {'question': 'Apakah benar Pasal 21 pada kode etik mahasiswa STTNF Etika mahasiswa dalam bidang penelitian',
  'ground_truth_answer': 'Tidak'},
 {'question': 'Apakah benar surat peringatan diberikan kepada mahasiswa yang mempunyai masa studi dibawah 6 tahun',
  'ground_truth_answer': 'Tidak'},
 {'question': 'Apakah benar di STTNF bahwa mahasiswa yang masa studinya melapui 8 semester akan diberlakukan ketentuan SPP Progresif',
  'ground_truth_answer': 'Ya'},
 {'question': 'Apakah benar minimal sks untuk lulus di sttnf adalah 148 SKS ',
  'ground_truth_answer': 'Ya'}]

In [306]:
df_laporan_pedoman_akademik, score_metric_pedoman_akademik = run_evaluation_binary(bot, dataset_rule)

Mulai Evaluasi


2026-02-20 09:49:46,375 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: query_from_academic_rule with {'query': 'syarat IPK minimal untuk lulus di STT-NF'}


2026-02-20 09:49:57,875 - INFO - POST http://localhost:9200/langchain_index/_search?_source_includes=metadata,text [status:200 duration:0.103s]
2026-02-20 09:50:14,525 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-20 09:50:28,830 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: query_from_academic_rule with {'query': 'Pasal 21 kode etik mahasiswa dalam bidang penelitian'}


...

2026-02-20 09:51:38,611 - INFO - POST http://localhost:9200/langchain_index/_search?_source_includes=metadata,text [status:200 duration:0.168s]
2026-02-20 09:51:52,005 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-20 09:52:09,227 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: query_from_academic_rule with {'query': 'ketentuan SPP Progresif untuk mahasiswa yang melakukan studi lebih dari 8 semester'}


2026-02-20 09:52:23,671 - INFO - POST http://localhost:9200/langchain_index/_search?_source_includes=metadata,text [status:200 duration:0.078s]
2026-02-20 09:52:38,645 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-20 09:52:55,255 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: query_from_academic_rule with {'query': 'syarat minimal sks untuk lulus di sttnf'}


2026-02-20 09:53:05,575 - INFO - POST http://localhost:9200/langchain_index/_search?_source_includes=metadata,text [status:200 duration:0.117s]
2026-02-20 09:53:21,060 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [307]:
df_laporan_pedoman_akademik

,No,Pertanyaan,Konteks Retrieval (Raw Data),Jawaban Generator (AI),Ekstrak,Target (GT),Status
0,1,Apakah benar untuk lulus di STTNF harus memili...,Ditemukan informasi berikut dari Pedoman Akade...,"Benar, mahasiswa harus memiliki IPK minimal 2...",Ya,Ya,✅ Sesuai
1,2,Apakah benar Pasal 21 pada kode etik mahasiswa...,Ditemukan informasi berikut dari Pedoman Akade...,"Maaf, terdapat kesalahan dalam pertanyaan And...",Tidak,Tidak,✅ Sesuai
2,3,Apakah benar surat peringatan diberikan kepada...,Ditemukan informasi berikut dari Pedoman Akade...,"Maaf, data tidak ditemukan. Surat peringatan ...",Tidak,Tidak,✅ Sesuai
3,4,Apakah benar di STTNF bahwa mahasiswa yang mas...,Ditemukan informasi berikut dari Pedoman Akade...,"Ya, benar. Di STT-NF, bagi mahasiswa yang mel...",Ya,Ya,✅ Sesuai
4,5,Apakah benar minimal sks untuk lulus di sttnf ...,Ditemukan informasi berikut dari Pedoman Akade...,"Benar, minimal SKS yang diperlukan untuk diny...",Ya,Ya,✅ Sesuai


In [308]:
score_metric_pedoman_akademik

{'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0}

In [309]:
df_laporan_pedoman_akademik.to_excel("df_laporan_pedoman_akademik.xlsx")

### Hybrid Data Evaluation

In [392]:
hybrid_dataset =[
    {
        "question":"Dari data akademik, apakah Rina Kusuma sudah mencapai SKS yang diperlukan untuk lulus dari STTNF",
        "ground_truth_answer": "Ya"
    },
    {
        "question": "Apakah benar Ini adalah semester Terakhir dari Yuni Ridwan berdasarkan syarat kelulusan",
        "ground_truth_answer": "Tidak"
        
    },
    {
        "question":"Dari data akademik mahasiswa apakah Agus Nasution akan dikenakan SPP Progresif berdasarkan pedoman akademik",
        "ground_truth_answer": "Ya"
    },
    {
        "question": "Berdasarkan pedoman akademik, apakah SKS dari Usman lubis telah memenuhi syarat untuk lulus",
        "ground_truth_answer": "Tidak"
    },

    {
        "question": "Berdasarkan pedoman akademik, apakah Opik Lubis bisa lulus dengan nilai IPK yang sekarang ",
        "ground_truth_answer": "Ya"
    }
]

In [393]:
df_laporan_hybrid_data, score_metric_hybrid = run_evaluation_binary(bot, hybrid_dataset)

Mulai Evaluasi


2026-02-20 13:55:10,255 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-20 13:55:19,788 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.048s]


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: get_student_academic_record with {'query': 'Rina Kusuma'}


2026-02-20 13:55:39,013 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-20 13:56:33,240 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: query_from_academic_rule with {'query': 'syarat kelulusan semester terakhir'}


...

2026-02-20 14:00:39,420 - INFO - POST http://localhost:9200/langchain_index/_search?_source_includes=metadata,text [status:200 duration:0.128s]
2026-02-20 14:01:15,076 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-20 14:02:26,965 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: query_from_academic_rule with {'query': 'syarat kelulusan IPK'}


2026-02-20 14:02:48,832 - INFO - POST http://localhost:9200/langchain_index/_search?_source_includes=metadata,text [status:200 duration:0.058s]
2026-02-20 14:02:48,842 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.007s]


🛠️ Executing: get_student_academic_record with {'query': 'Opik Lubis'}


2026-02-20 14:03:24,307 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [395]:
df_laporan_hybrid_data

,No,Pertanyaan,Konteks Retrieval (Raw Data),Jawaban Generator (AI),Ekstrak,Target (GT),Status
0,1,"Dari data akademik, apakah Rina Kusuma sudah m...",Ditemukan informasi berikut dari data akademik...,"YA, Rina Kusuma sudah mencapai IPS yang diperl...",Tidak,Ya,❌ Meleset
1,2,Apakah benar Ini adalah semester Terakhir dari...,Ditemukan informasi berikut dari Pedoman Akade...,"- Maaf, data tidak ditemukan. Mohon periksa na...",Tidak,Tidak,✅ Sesuai
2,3,Dari data akademik mahasiswa apakah Agus Nasut...,Ditemukan informasi berikut dari data akademik...,"YA, Agus Nasution akan dikenakan SPP Progresif...",Ya,Ya,✅ Sesuai
3,4,"Berdasarkan pedoman akademik, apakah SKS dari ...",Ditemukan informasi berikut dari data akademik...,[EKSTRAKSI DATA MAHASISWA]\n- NIM: 178\n- Pro...,Tidak,Tidak,✅ Sesuai
4,5,"Berdasarkan pedoman akademik, apakah Opik Lubi...",Ditemukan informasi berikut dari Pedoman Akade...,"Ya, Opik Lubis dapat lulus.",Ya,Ya,✅ Sesuai


In [396]:
df_laporan_hybrid_data.to_excel("laporan_hybrid_3.xlsx")

In [397]:
score_metric_hybrid

{'Accuracy': 0.8, 'Precision': 1.0, 'Recall': 0.67}